Notebook that calculates Reynolds number given a snapshot of velocity and manually defined length scales

In [1]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# ------------------------------------------------------------
# IMPORT PROJECT MODULES
# ------------------------------------------------------------
NB_DIR = Path.cwd()

# ocean_analysis.py in OGGCM/scripts
OCEAN_SCRIPT_DIR = NB_DIR.parents[1] / "OGGCM" / "scripts"
if str(OCEAN_SCRIPT_DIR) not in sys.path:
    sys.path.append(str(OCEAN_SCRIPT_DIR))

import ocean_analysis as oa
importlib.reload(oa)

# shcherbina_utils.py in z.flow_postprocessing/scripts
UTILS_PATH = (NB_DIR / "../scripts").resolve()
if str(UTILS_PATH) not in sys.path:
    sys.path.insert(0, str(UTILS_PATH))

MODULE_NAME = "shcherbina_utils"
shu = importlib.import_module(MODULE_NAME)
importlib.reload(shu)

ModuleNotFoundError: No module named 'ocean_analysis'

In [3]:

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
DATA_FILE = (NB_DIR / "../data/input/run_boyd_SGS_L1.nc").resolve()
OUT_DIR = (NB_DIR / "../results/reynolds_run_boyd_SGS_L1").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Could not find data file:\n{DATA_FILE}")

SAVE = False

DX = 500.0
DY = 500.0

SELECTED_TIME_INDEX = 6
DAY_PER_INDEX = 0.5

RE_LEVEL_INDEX = 0

# Viscosity reference.
# For harmonic constant viscosity, this can be viscAh.
# For Leith/Smagorinsky, interpret as background/reference viscosity unless
# you have a diagnosed effective viscosity.

NU_REF = 1.0  # m2/s

# Domain subdivision, only used for a manually defined "box" length scale.
N_SQUARES_X = 4
N_SQUARES_Y = 4

# EDS / spectral peak settings
RUN_EDS = True
EDS_N_BINS = 16
EDS_REMOVE_MEAN = True

# Avoid selecting near-grid scales as the spectral peak.
# With DX=500 m, 4*DX = 2 km is a conservative lower wavelength bound.
SPECTRAL_MIN_WAVELENGTH_M = 4 * DX
SPECTRAL_MAX_WAVELENGTH_M = None


# ------------------------------------------------------------
# OPEN DATA AND FIND DIMENSIONS
# ------------------------------------------------------------
ds = xr.open_dataset(DATA_FILE, chunks="auto")

required_vars = ["UVEL", "VVEL"]
for var in required_vars:
    if var not in ds:
        raise KeyError(f"Required variable '{var}' not found in dataset.")

time_dim = shu.find_dim(ds.dims, ["T", "time", "iter"])
z_dim = shu.find_dim(ds["UVEL"].dims, ["Z", "Zmd", "k", "depth"])

nt = ds.sizes[time_dim]
nz = ds.sizes[z_dim]

if not (0 <= SELECTED_TIME_INDEX < nt):
    raise IndexError(f"SELECTED_TIME_INDEX={SELECTED_TIME_INDEX} out of range 0..{nt-1}")

if not (0 <= RE_LEVEL_INDEX < nz):
    raise IndexError(f"RE_LEVEL_INDEX={RE_LEVEL_INDEX} out of range 0..{nz-1}")

time_days = shu.get_time_days(SELECTED_TIME_INDEX, DAY_PER_INDEX)

print(f"DATA_FILE           : {DATA_FILE}")
print(f"time_dim            : {time_dim}")
print(f"z_dim               : {z_dim}")
print(f"n time steps        : {nt}")
print(f"n vertical levels   : {nz}")
print(f"selected snapshot   : index {SELECTED_TIME_INDEX} -> t = {time_days:.1f} days")
print(f"selected level      : k = {RE_LEVEL_INDEX}")
print(f"NU_REF              : {NU_REF:g} m2/s")


# ------------------------------------------------------------
# EXTRACT CENTERED VELOCITIES
# ------------------------------------------------------------
u_c, v_c = shu.get_centered_uv_snapshot(
    ds,
    time_dim=time_dim,
    z_dim=z_dim,
    time_index=SELECTED_TIME_INDEX,
    level_index=RE_LEVEL_INDEX,
)

ny, nx = u_c.shape

print(f"centered velocity shape: ny={ny}, nx={nx}")


# ------------------------------------------------------------
# MANUALLY CHOSEN LENGTH SCALES
# ------------------------------------------------------------
L_DOMAIN_X = nx * DX
L_DOMAIN_Y = ny * DY

length_scales = {
    "dx": DX,
    "5dx": 5 * DX,
    "10dx": 10 * DX,
    "box": (nx / N_SQUARES_X) * DX,
    "domain_min": min(L_DOMAIN_X, L_DOMAIN_Y),
}

print("\nLength scales:")
for name, L in length_scales.items():
    print(f"  {name:10s}: {L:10.1f} m = {L/1000:7.2f} km")


# ------------------------------------------------------------
# METHOD 1: TOTAL VELOCITY RMS WITH FIXED LENGTH SCALES
# ------------------------------------------------------------
Re_method1 = shu.reynolds_fixed_length_scales(
    u_c,
    v_c,
    length_scales_m=length_scales,
    nu=NU_REF,
    remove_mean=False,
)

print("\nMethod 1: total velocity RMS")
display(pd.DataFrame([Re_method1]).T)


# ------------------------------------------------------------
# METHOD 2: EDDY / ANOMALY VELOCITY RMS WITH FIXED LENGTH SCALES
# ------------------------------------------------------------
Re_method2 = shu.reynolds_fixed_length_scales(
    u_c,
    v_c,
    length_scales_m=length_scales,
    nu=NU_REF,
    remove_mean=True,
    anomaly_method="domain_mean",
)

print("\nMethod 2: anomaly velocity RMS")
display(pd.DataFrame([Re_method2]).T)


# ------------------------------------------------------------
# METHOD 3: PEAK SPECTRAL-DENSITY LENGTH SCALE FROM EDS
# ------------------------------------------------------------
if RUN_EDS:
    eds = oa.calculate_EDS_init(
        filepath=DATA_FILE,
        snapshot_index=SELECTED_TIME_INDEX,
        x_res=DX,
        y_res=DY,
        n_bins=EDS_N_BINS,
        remove_mean=EDS_REMOVE_MEAN,
    )

    Re_method3 = shu.reynolds_peak_spectral_scale(
        u_c,
        v_c,
        eds_ds=eds,
        nu=NU_REF,
        remove_mean=True,
        min_wavelength_m=SPECTRAL_MIN_WAVELENGTH_M,
        max_wavelength_m=SPECTRAL_MAX_WAVELENGTH_M,
    )

    print("\nMethod 3: spectral peak length scale")
    display(pd.DataFrame([Re_method3]).T)

else:
    eds = None
    Re_method3 = None


# ------------------------------------------------------------
# ONE-CALL SUMMARY TABLE
# ------------------------------------------------------------
re_summary = shu.summarize_reynolds_strategies(
    u_c,
    v_c,
    dx=DX,
    dy=DY,
    nu=NU_REF,
    eds_ds=eds,
    box_length_m=length_scales["box"],
    spectral_min_wavelength_m=SPECTRAL_MIN_WAVELENGTH_M,
    spectral_max_wavelength_m=SPECTRAL_MAX_WAVELENGTH_M,
    anomaly_method="domain_mean",
)

re_df = pd.DataFrame([re_summary])
display(re_df.T)


# ------------------------------------------------------------
# OPTIONAL SAVE
# ------------------------------------------------------------
if SAVE:
    out_csv = OUT_DIR / f"reynolds_snapshot_t{SELECTED_TIME_INDEX:04d}_k{RE_LEVEL_INDEX}.csv"
    re_df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")


print("\nNotebook variables now available:")
print("  ds")
print("  u_c, v_c")
print("  Re_method1, Re_method2, Re_method3")
print("  re_df")
if RUN_EDS:
    print("  eds")

NameError: name 'shu' is not defined